<a href="https://colab.research.google.com/github/Jack3690/pydol/blob/main/notebooks/Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install  "pydol[jwst] @ git+https://github.com/Jack3690/pydol" -q

In [ ]:
# Optional
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
cd /content/drive/MyDrive/rsp_jwst_school

# **Imports**

In [ ]:
from glob import glob
import astropy.io.fits as fits
import json
import os, sys
import astroquery
from astroquery.mast import Observations
import pandas as pd

from astropy.table import Table,unique, vstack
from astropy.wcs import WCS
import numpy as np
import multiprocessing as mp
import matplotlib.pyplot as plt
import json
from astropy.coordinates import angular_separation
import astropy.units as u
from pydol.pipeline.jwst import jpipe
import multiprocessing as mp
from pydol.photometry.scripts import cmdtools
from pydol.photometry import nircam

# **Data**

**Observation ID format**

jw\<ppppp>\<ooo><vvv>\_\<gg>\<s>\<aa>\_\<eeeee>(-<"seg"NNN>)\_\<detector>\_\<prodType>.fits

where

ppppp: program ID number

ooo: observation number

vvv: visit number

gg: visit group

s: parallel sequence ID (1=prime, 2-5=parallel)

aa: activity number (base 36)

eeeee: exposure number

segNNN: the text “seg” followed by a three-digit segment number (optional)

detector: detector name (e.g. ‘nrca1’, ‘nrcblong’, ‘mirimage’)

prodType: product type identifier (e.g. ‘uncal’, ‘rate’, ‘cal’)

In [ ]:
#Example data for F200W filter for NGC628
filters = ['F200W']
obs_table = Observations.query_criteria(proposal_id=["1783"],objectname="NGC628",
                                        filters= filters,
                                        dataRights='PUBLIC')[:3]

t = [Observations.get_product_list(obs) for obs in obs_table]
files = unique(vstack(t), keys='productFilename')

In [ ]:
# Selecting only images from detector a1
ind = np.array([True if 'a1' in t else False for t in files['obs_id'] ])


In [ ]:
mast_dir = 'mast:jwst/product' # Download from MAST

for filt in filters:
  data_dir = f'data/{filt}/stage0/'  # save downloaded data

  os.makedirs(data_dir, exist_ok=True)
  manifest = Observations.download_products(
              files[ind],
              dataRights = 'PUBLIC',
              productSubGroupDescription = 'UNCAL',
              download_dir = data_dir,
              flat = True,
              )

# **Pipeline**

In [ ]:
os.makedirs('CRDS', exist_ok=True)

In [ ]:
input_files = glob(f'data/F200W/stage0/*')
out_dir     = 'data/F200W/'
jwst_data   = jpipe(input_files, out_dir,crds_dir='CRDS/', filter='F115W',
                    crds_context='jwst_1464.pmap', n_cores=1)
jwst_data()

# **DOLPHOT**

In [ ]:
#Example data for F200W filter for NGC628
filters = ['F200W']
obs_table = Observations.query_criteria(proposal_id=["1783"],objectname="NGC628",
                                        filters= filters,
                                        dataRights='PUBLIC')[:3]

t = [Observations.get_product_list(obs) for obs in obs_table]
files = unique(vstack(t), keys='productFilename')

In [ ]:
# Selecting only images from detector a1
ind = np.array([True if 'a1' in t else False for t in files['obs_id'] ])


In [ ]:
mast_dir = 'mast:jwst/product' # Download from MAST

for filt in filters:
  data_dir = f'data/{filt}/stage2/'  # save downloaded data

  os.makedirs(data_dir, exist_ok=True)
  manifest = Observations.download_products(
              files[ind],
              dataRights = 'PUBLIC',
              productSubGroupDescription = 'CAL',
              download_dir = data_dir,
              flat = True,
              )

In [ ]:
mast_dir = 'mast:jwst/product' # Download from MAST

for filt in filters:
  data_dir = f'data/photometry/'  # save downloaded data

  os.makedirs(data_dir, exist_ok=True)
  manifest = Observations.download_products(
              files[files['calib_level']==3],
              dataRights = 'PUBLIC',
              productSubGroupDescription = 'I2D',
              download_dir = data_dir,
              flat = True,
              )

## **Setting up**

In [ ]:
!wget http://americano.dolphinsim.com/dolphot/dolphot3.1.tar.gz
!wget http://americano.dolphinsim.com/dolphot/dolphot3.1.NIRCAM.tar.gz
!wget http://americano.dolphinsim.com/dolphot/nircam_F200W.tar.gz

In [ ]:
! tar -xzf dolphot3.1.tar.gz
! tar -xzf dolphot3.1.NIRCAM.tar.gz
!tar --transform='s|^\./dolphot2\.0|./dolphot3.1|' \
    -xzf nircam_F200W.tar.gz

In [ ]:
with open('dolphot3.1/Makefile') as f:
  dat = f.readlines()

for n, d in enumerate(dat):
  if 'USENIRCAM' in d:
    break
dat[n]=dat[n][1:]

with open('dolphot3.1/Makefile','w') as f:
  f.writelines(dat)

In [ ]:
cd dolphot3.1

In [ ]:
!make

In [ ]:
cd ..

In [ ]:
new_path = "/content/drive/MyDrive/rsp_jwst_school/dolphot3.1/bin/"

# Append it to the existing PATH variable
os.environ["PATH"] += os.pathsep + new_path

In [ ]:
!nircammask

In [ ]:
!pwd

In [ ]:
for j in ['a1']:
    crf_files = glob(f'data/F200W/stage2/*{j}*_cal.fits')
    nircam.nircam_phot(crf_files,
                filter='f200w',
                output_dir= f'data/photometry/{j}',
                ref_img_path='data/photometry/jw01783-o004_t008_nircam_clear-f200w_i2d',
                quality_filters = 'f200w')

In [ ]:
tab = Table.read('data/photometry/f200w_photometry_filt.fits')